# NB1 · Reaching the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What the workshop builds

Across six notebooks a working clinical decision support system is written. The system
is a computer program, and each notebook adds a layer to it.

| Notebook | Layer added |
|---|---|
| NB1 | Reaching the data and taking a first look |
| NB2 | Cleaning, preparing the information, splitting into training and test groups |
| NB3 | Building the model, teaching it and measuring how well it does |
| NB4 | The layer that justifies the model's decision |
| NB5 | Safety guardrails and the compliance report |
| NB6 | An interface used from a web browser |

**You do not need to know Python.** You will not write the code. Each step gives you a
prompt; you pass it to a generative AI tool (ChatGPT, Claude, Gemini), then paste the
code it returns into the blank cell and run it.

Understanding what the code does is expected. A short Python note follows each step and
explains the structures you will have seen in the code you received.


## How the notebook works

Each prompt ends with a section headed EXPECTED RESULT. It states what the code must
produce: which name will hold which piece of information. The check cell that follows
the paste cell tests exactly that.

When a check reports a shortcoming, return the code to the AI tool, state what the check
reported and have it regenerated. Not succeeding on the first attempt is normal.

The first line of each paste cell reads `#@cdss step_name`. **Do not delete that line.**
Paste your code below it. At the end of the notebook every step is collected into a
single block, which you carry to the next notebook.


## Setup

The cell below downloads the check helper. It is the only code handed to you in this
workshop; everything else you will have generated.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Step 1 · The start of the program

Every program begins with a few lines of preparation. Two things happen: the ready made
toolkits are called in, and the values that will not change during the program are
written once.

Our system has four such values. A decision is produced **six hours** after the patient
enters intensive care; information recorded up to that moment may be used, and anything
after it may not. The condition we target is a stay exceeding **three days**. The address
of the data and a number for reproducibility are also written here.


### Prompt 1

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section of the program.

Call in the standard toolkits needed for handling data and for machine learning.

Then define the four fixed values of the program, writing beside each what it means:
  - Decision moment: six hours after the patient enters intensive care.
  - Target threshold: an intensive care stay exceeding three days.
  - Reproducibility number: 42.
  - Address of the data: https://physionet.org/files/mimic-iv-demo/2.2

Name these DECISION_WINDOW_HOURS, TARGET_THRESHOLD_DAYS, RANDOM_SEED and DATA_ROOT.
Write them at the top.

Print the versions of the toolkits you used.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
After the cell has run, these names must be available:
  pd, np, DECISION_WINDOW_HOURS, TARGET_THRESHOLD_DAYS, RANDOM_SEED, DATA_ROOT
```


In [ ]:
#@cdss hazirlik
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_defined('pd', 'np', 'DECISION_WINDOW_HOURS', 'TARGET_THRESHOLD_DAYS',
                  'RANDOM_SEED', 'DATA_ROOT')


### Python note · What is in the code you received

At the top of your code you will see lines such as `import pandas as pd`. A **library**
is a ready made toolkit written by others. `import` calls it into the program, and
`as pd` gives it a short name, so `pd` can be written instead of `pandas` from then on.
`pandas` is for working with tables and `numpy` for calculating with numbers.

The line `DECISION_WINDOW_HOURS = 6` defines a **variable**. Writing it in capitals tells
the reader that this value will not change during the program. Python does not enforce
this; it is a convention.

Keeping these values at the top matters. A `6` buried in the middle of the code becomes,
six months later, a value nobody remembers the meaning of. In a clinical system threshold
values are subject to audit and have to be visible in one place.

`RANDOM_SEED` serves this purpose: parts of machine learning involve randomness. Without
a fixed seed the program gives a slightly different result on each run, and which change
caused what can no longer be traced.


In [ ]:
# This cell is supplied. It prints the values you defined.
print('Decision moment :', DECISION_WINDOW_HOURS, 'hours')
print('Target threshold:', TARGET_THRESHOLD_DAYS, 'days')
print('Data address    :', DATA_ROOT)


---

## Step 2 · Bringing in the data

The shared scenario uses the MIMIC-IV demo dataset. It covers one hundred patients, is
openly accessible, requires no password and is read directly from the web.

The data sits in three separate files. I have written into the prompt what each file
holds. Do not expect the AI tool to know this on its own; it fills what it does not know
by guessing, and the guess is usually wrong. **You are the one who tells the tool what
your data contains.** This is the habit from the workshop that will serve you most.

If your data is of a different type, choose prompt 2b, 2c or 2d. All four produce the
same result, so the steps that follow proceed identically.


### Prompt 2a · Hospital records (shared scenario)

```
Bring the data of patients admitted to intensive care into the program. The data is
openly accessible on the web and its address is defined above as DATA_ROOT.

There are three files, all compressed table files:
  {DATA_ROOT}/hosp/patients.csv.gz
      patient identifier (subject_id), sex (gender), age (anchor_age)
  {DATA_ROOT}/hosp/admissions.csv.gz
      patient identifier (subject_id), hospital admission number (hadm_id),
      type of admission (admission_type), insurance (insurance)
  {DATA_ROOT}/icu/icustays.csv.gz
      patient identifier (subject_id), hospital admission number (hadm_id),
      intensive care stay number (stay_id), which unit (first_careunit),
      time of entry (intime), time of exit (outtime),
      how many days the patient stayed in intensive care (los)

Do the following:
1. Start from the intensive care stays. Remove the patients who left before the decision
   moment, that is, those who stayed fewer than DECISION_WINDOW_HOURS hours.
2. Create the condition we will predict: 1 if the stay exceeds TARGET_THRESHOLD_DAYS,
   0 if not. Name it target.
3. Rename the patient identifier to patient_id.
4. Bring in the patient's sex, age, admission type and insurance from the other two
   files. The number of rows must not change during this; warn me if it does.
5. This item matters a great deal: remove from the result how many days the patient
   stayed (los) and when they left (outtime). Both are known only after the patient
   leaves and are not available at the decision moment.

Put all of this inside a piece of work named load_raw_data. Write an explanation of what
it does at its start. Then run it and keep the result under the name cohort. Show the
first five rows of the cohort.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named load_raw_data that returns a table.
There must be a table named cohort.
That table must contain patient_id, stay_id, target, gender, anchor_age,
admission_type, insurance and first_careunit.
That table must NOT contain los or outtime.
```


### Prompt 2b · Medical imaging

```
Prepare data for a clinical decision support system that works on images.

My problem: [write your own problem here in one sentence]

Use a suitable set from an openly accessible medical image collection; if none fits,
generate artificial images that contain the sign we are looking for. Let the same patient
have more than one image, as is the case with real clinical data.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> patient identifier, the same patient may have several rows
  image      -> the image itself
  target     -> 0 or 1
```


### Prompt 2c · Physiological signal

```
Prepare data for a clinical decision support system that works on a continuously
measured physiological signal.

My problem: [write your own problem here in one sentence]

Use an openly accessible signal collection if a suitable one exists, otherwise generate
an artificial signal. Divide the signal into segments of fixed length and state in a
comment why you chose that length. Allow more than one segment per patient.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> patient identifier
  signal     -> the signal segment
  target     -> 0 or 1
```


### Prompt 2d · Clinical text

```
Prepare data for a decision support system that works on clinical notes.

My problem: [write your own problem here in one sentence]

Generate artificial clinical notes. Give them the features that make real notes hard:
abbreviations, negation, expressions of uncertainty, templated sentences repeated in
every note, and sections copied forward from earlier notes. The condition we look for
must not be recoverable from a single word.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> patient identifier
  note       -> the clinical note
  target     -> 0 or 1
```


In [ ]:
#@cdss veri_yukleme
# Paste the generated code below this line.


### Check 2

Two checks run. The first looks at whether the piece of work runs, the second at whether
the resulting table matches the expected result. If you are not on the shared scenario,
change the column names in the second cell to match your own data.


In [ ]:
kit.check_function('load_raw_data', call_with=((), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(
    cohort, name='cohort',
    required=['patient_id', 'target'],
    forbidden=['los', 'outtime'],
    min_rows=30,
)


### Python note · Tables and pieces of work

In the code you received you will see lines such as `pd.read_csv(...)` and an object
named `cohort`. That object is a **table**; in the Python world it is called a DataFrame.
Think of an Excel sheet: rows hold records and columns hold fields. In our table each row
is one intensive care stay and each column a piece of information about that stay.

The line `def load_raw_data():` defines a **function**. A function is a piece of work with
a name. It is defined once and can be run as often as needed. Putting work into a function
brings three benefits: the work lives in one place, it can be repeated and it can be
tested.

The text in triple quotes just below the function is its **docstring**. Unlike a comment,
it can be read while the program runs, as the next cell shows.

`return` gives the result back. Without it the result stays inside the function and
nobody can use it.


In [ ]:
# Read the explanation written into the function.
help(load_raw_data)


In [ ]:
# The size of the table and its first rows.
print('Rows and columns:', cohort.shape)
print('Column names:', list(cohort.columns))
cohort.head()


### Why two columns were removed

The fifth item of the prompt asked for `los` and `outtime` to be removed. The reason is
this: how many days a patient stayed is known only after they leave. Give that to the
model and it performs almost perfectly, because we have told it the answer.

Once the system is deployed in a hospital, that column is empty at the decision moment
and the system is of no use. This is called **leakage**. It is the most common error in
clinical code written with generative AI; it raises no error, proceeds silently, and looks
favourable at first because it improves the result.

Rather than trying to catch leakage afterwards, we prevented it at the prompt stage. The
`forbidden` list in the check cell tested it as well.


---

## Step 3 · A first look at the data

Before moving to a model you have to know the table in front of you. In this step you
will have a piece of work written that summarises it.


### Prompt 3

```
Write a piece of work that summarises the cohort table. Name it summarise_data and let
it summarise whatever table it is given.

Have it print:
  - how many rows there are,
  - how many distinct patients there are,
  - the rate at which the target condition occurs, as a percentage.

Have it also give back a summary table with one row per column, containing:
  column       -> the name of the column
  type         -> the kind of information in it
  missing_rate -> how much of the column is empty
  distinct     -> how many different values it holds

Then run it on cohort, keep the result under the name summary and show it.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named summarise_data that returns a table.
There must be a table named summary containing column, type, missing_rate and distinct.
```


In [ ]:
#@cdss veri_kesfi
# Paste the generated code below this line.


### Check 3


In [ ]:
kit.check_function('summarise_data', call_with=((cohort,), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(summary, name='summary',
                required=['column', 'type', 'missing_rate', 'distinct'])


### Python note · Printing and giving back

The prompt asked for two different things: some figures printed, and a summary table
given back. The difference matters.

`print(...)` writes to the screen. What it writes stays on the screen and the program
cannot use it again. It cannot be tested either; the check cell cannot see text on a
screen.

`return ...` gives the result back. A table given back can be stored, tested and used in
later steps. That is why we had the summary table returned; you will add it to the
compliance report in NB5.

As a rule: print a result you only want a person to see, and return a result you want the
program to use.


### Reading the summary

Look at three things.

**Is the patient count lower than the row count?** If so, some patients have more than one
stay. You will have to take that into account when splitting the data in NB2.

**What is the rate of the target condition?** This is how often the condition we seek
occurs in the cohort. It will be the single most decisive figure when performance is
interpreted in NB3.

**Which columns are largely empty?** In NB2 you will decide how missing values are filled.
For now simply note them.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB2.

The block is also saved as `cdss_nb1.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb1.py')


## What this notebook did

The first layer of the system was written. The program now prepares itself, takes the
data from the web and summarises the table it holds.

Three habits were formed while the code was produced. What the data contains was stated
to the tool explicitly, and it was not left to guess. An expected result was written at
the end of every prompt, and the code that arrived was tested against it. Information not
available at the decision moment was removed while the data was still being loaded.

In NB2, cleaning, preparation of the information and the split into training and test
groups are added on top of this code.
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The
MIMIC-IV demo data comes from a single hospital in the United States and does not
represent an intensive care population elsewhere. The material is for teaching.
